In [0]:
from pyspark.sql.functions import col, sum, avg, count, max, min, round
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# ---- 1. SEED DATA SOURCE: Base Employee Dataframe
emp_data = [
    (1, "Rahul", "Mumbai", 45000, "Engineering"),
    (2, "Priya", "Bangalore", 62000, "Analytics"),
    (3, "Arjun", "Delhi", 38000, "Engineering"),
    (4, "Sneha", "Bangalore", 75000, "Analytics"),
    (5, "Vikram", "Chennai", 55000, "Engineering"),
    (6, "Meera", "Mumbai", 82000, "Analytics"),
]

emp_schema = StructType([
   StructField("emp_id", IntegerType(), False),
   StructField("name", StringType(), True),
   StructField("city", StringType(), True),
   StructField("salary", IntegerType(), True),
   StructField("department", StringType(), True),
])

df_emp = spark.createDataFrame(emp_data, emp_schema)
df_emp.show()

# ---- 2. AGGREGATION: Summary Calculation 
dept_summary = df_emp.groupBy("department").agg(
    count("emp_id").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary")
)

print("=== Department Salary Aggregation Summary ===")
dept_summary.show(truncate=False)

# --- 3. REUSE TARGET: Department Lookup Table
dept_data = [
    ("Engineering", "Bangalore", "Ravi Kumar"),
    ("Analytics", "Mumbai", "Ananya Patel"),
]

dept_schema = StructType([
    StructField("dept_name", StringType(), True),
    StructField("dept_hq_city", StringType(), True),
    StructField("dept_head", StringType(), True),
])

df_dept = spark.createDataFrame(dept_data, dept_schema)
df_dept.show()

# ---- 4. INNER JOIN: Combining Datasets 
df_joined = df_emp.join(
    df_dept,
    df_emp["department"] == df_dept["dept_name"],
    "inner"
)

print("==== Inner Join Results ====")
df_joined.select("name", "city", "salary", "department", "dept_head").show(truncate=False)

# ---- 5. LEFT OUTER JOIN: Master-Detail Preserving
df_left = df_emp.join(
    df_dept,
    df_emp["department"] == df_dept["dept_name"],
    "left_outer"
)

print("==== Left Join Results ====")
df_left.show(truncate=False)